# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `train.jsonl` (로컬 `D:\calendar-agent\data\processed\train.jsonl`)
   - `val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calendar-agent\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-messages` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/datasets/sooryongbyun/calendar-messages/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

## 0. GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, force-syncing to origin/main…')
    !cd calendar-agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + Colab/Kaggle 호환 cleanup

Colab과 동일 — install + torchao 제거.

In [ ]:
# 설치. ⚠ Kaggle 사전설치 RAPIDS(cudf/cuml/dask-cuda)가 numba/cuda-core를 옛 버전에 고정해 둬서
#   "X requires ... which is incompatible" 충돌 경고가 뜨지만, 우리 학습은 RAPIDS 미사용 → 무해.
#   아래 grep으로 그 충돌 보고 줄만 숨김(실제 설치 실패 메시지는 패턴이 달라 그대로 보임).
!pip install -q -e .[train] 2>&1 | grep -vE "which is incompatible|pip's dependency resolver does not|source of the following dependency"
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ["WANDB_DISABLED"] = "true"
# CUDA_VISIBLE_DEVICES 설정 안 함 — DDP(torchrun)가 2 GPU를 다 써야 함.
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. 데이터 확인 (repo에 포함 — clone으로 따라옴, Kaggle 데이터셋 불필요)

In [ ]:
# 데이터는 repo(git)에 포함 → clone으로 따라옴 (train 2245 = r6 데이터, golden 51).
# 예전엔 Kaggle 데이터셋에서 복사했으나, 이제 git이 단일 출처. 데이터셋 첨부 불필요.
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p}: {n} records")

## 5. 학습 실행

약 2040건 × 3 epochs ≈ 192 step. T4/P100 기준 ~1시간.

**Kaggle 세션 한도**: 9시간/세션 — 우리 학습은 1시간이라 무관.

In [ ]:
# 비교할 모델 config 선택 (0.5B vs 1.5B — 동일 데이터)
#   0.5B: configs/train.yaml   /   1.5B: configs/train_1_5b.yaml
CONFIG = 'configs/train.yaml'   # ← 1.5B 학습 시 'configs/train_1_5b.yaml'로 바꿔 이 셀부터 다시 실행
print('학습 config:', CONFIG)
# DDP: T4 2개를 진짜로 병렬 사용 (torchrun, DataParallel 아님)
!torchrun --nproc_per_node=2 scripts/train_lora.py --config {CONFIG}

## 5.5 학습 직후 평가 (Kaggle GPU, 선택)

merge 후 골든셋으로 바로 평가해 `time_match_rate`/`final_score`를 확인한다. 로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠르다. 결과 json은 아래 6번 zip에 포함된다.

In [ ]:
# CONFIG에서 모든 경로 자동 인식 — base(0.5B/1.5B)는 model_config의 hf_id에서 읽음.
import yaml, os
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']                          # Qwen2.5-0.5B / 1.5B 자동
LORA_DIR = _cfg['output_dir']                  # 예: models/lora/r12-qwen0.5b
NAME = os.path.basename(LORA_DIR)              # r12-qwen0.5b
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/real_golden.jsonl')
ZIP = f'/kaggle/working/lora_{NAME}.zip'       # 모델별로 구분 (0.5b / 1.5b)
print(f'base={BASE}\nlora={LORA_DIR}\ngolden={GOLDEN}\nzip={ZIP}')

# merge → 실 골든셋 평가 (Kaggle GPU)
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 6. 결과 패키징 (Kaggle Output 자동 노출)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} {CONFIG} configs/lora.yaml {_cfg['model_config']} {EVAL_JSON}
!ls -la {ZIP}

## 다운로드 방법

**FileLink는 Kaggle 프록시에서 동작하지 않는다(404 남).** 아래 방법을 쓸 것:

1. **우상단 `Save Version` → `Quick Save`** (재실행 안 함, 현재 `/kaggle/working` 스냅샷) → 저장되면 버전 열기 → **Output** 탭에서 `lora_<라운드>.zip` 다운로드  ← 가장 확실
2. 또는 에디터 우측 **Output(`/kaggle/working`) 파일 패널**에서 `lora_<라운드>.zip` 다운로드 (안 보이면 새로고침)
3. 또는 로컬에서 Kaggle API: `kaggle kernels output <user>/<notebook-slug> -p .`

학습 직후 평가(아래 5.5)를 돌렸다면 `logs/eval_<라운드>-qwen.json`의 `time_match_rate` / `final_score`를 셀 출력에서 바로 확인할 수 있다.

In [ ]:
# 참고: FileLink는 Kaggle에서 종종 404. 위 '다운로드 방법'의 Quick Save -> Output 사용 권장.
from IPython.display import FileLink
FileLink(ZIP)